# MinIO S3 Data Explorer - Gold Layer

In [ ]:
# Cài đặt các thư viện cần thiết
import subprocess
import sys

# Tất cả packages cần thiết cho notebook
packages = [
    'minio',
    'pandas',
    'pyarrow',
    'confluent-kafka',
    'fastavro',
    'cachetools',
    'httpx',
    'attrs',
    'authlib',
    'requests'
]

print("[INFO] Cài đặt thư viện...\n")
failed_packages = []

for package in packages:
    try:
        # Thử import package (handle package name khác import name)
        if package == 'confluent-kafka':
            __import__('confluent_kafka')
        elif package == 'pyarrow':
            __import__('pyarrow')
        elif package == 'httpx':
            __import__('httpx')
        elif package == 'cachetools':
            __import__('cachetools')
        else:
            __import__(package)
        print(f"[OK] {package:<25} - đã có sẵn")
    except ImportError:
        print(f"[INSTALL] Đang cài {package:<25}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
            print(f"[OK] {package:<25} - cài xong")
        except Exception as err:
            print(f"[ERROR] {package:<25} - LỖI: {err}")
            failed_packages.append(package)

if failed_packages:
    print(f"\n[WARN] Các package cài đặt thất bại: {failed_packages}")
else:
    print("\n[OK] Tất cả thư viện sẵn sàng!")

[INFO] Cài đặt thư viện...

[OK] minio                     - đã có sẵn
[OK] pandas                    - đã có sẵn
[OK] pyarrow                   - đã có sẵn
[OK] confluent-kafka           - đã có sẵn
[OK] fastavro                  - đã có sẵn
[OK] cachetools                - đã có sẵn
[OK] httpx                     - đã có sẵn
[OK] attrs                     - đã có sẵn
[OK] authlib                   - đã có sẵn
[OK] requests                  - đã có sẵn

[OK] Tất cả thư viện sẵn sàng!


In [1]:
import os
import io
from minio import Minio
import pandas as pd
import pyarrow.parquet as pq
from datetime import datetime

print("[INFO] Imports hoàn tất!")

[INFO] Imports hoàn tất!


## MinIO Configuration for Gold Layer Exploration

In [3]:
MINIO_CONFIG = {
    'endpoint': 'localhost:9000',
    'access_key': 'minioadmin',
    'secret_key': 'minioadmin',
    'secure': False,
    'region': 'ap-southeast-1'
}

BUCKET_NAME = 'data-lake'
GOLD_PATH_PREFIX = 'warehouse/gold/'

try:
    minio_client = Minio(
        MINIO_CONFIG['endpoint'],
        access_key=MINIO_CONFIG['access_key'],
        secret_key=MINIO_CONFIG['secret_key'],
        secure=MINIO_CONFIG['secure'],
        region=MINIO_CONFIG['region']
    )
    
    # Kiểm tra kết nối
    if minio_client.bucket_exists(BUCKET_NAME):
        print(f"[SUCCESS] Kết nối MinIO thành công! Bucket '{BUCKET_NAME}' đã sẵn sàng.")
    else:
        print(f"[ERROR] Bucket '{BUCKET_NAME}' không tồn tại.")
    
except Exception as e:
    print(f"[ERROR] Kết nối MinIO thất bại: {e}")

[SUCCESS] Kết nối MinIO thành công! Bucket 'data-lake' đã sẵn sàng.


## 📊 Overview of the tables in the Gold layer

There are two main mart tables in the Gold layer:
1. **mart_sales_overview**: sales details table (denormalized).
2. **mart_customer_lifetime_value**: customer behavior table.

In [4]:
def list_gold_tables():
    try:
        objects = minio_client.list_objects(BUCKET_NAME, prefix=GOLD_PATH_PREFIX, recursive=True)
        tables = {}
        for obj in objects:
            parts = obj.object_name.split('/')
            if len(parts) >= 3:
                table_name = parts[2]
                if table_name not in tables:
                    tables[table_name] = {'count': 0, 'size_mb': 0}
                tables[table_name]['count'] += 1
                tables[table_name]['size_mb'] += obj.size / (1024 * 1024)
        return tables
    except Exception as e:
        print(f"Lỗi: {e}")
        return {}

gold_tables = list_gold_tables()
print("\nBảng Gold hiện có trong MinIO:")
print("=" * 50)
for name, info in gold_tables.items():
    print(f"- {name:<30} | {info['count']:>3} objects | {info['size_mb']:>6.2f} MB")


Bảng Gold hiện có trong MinIO:
- mart_customer_lifetime_value   |   6 objects |   0.02 MB
- mart_sales_overview            |   6 objects |   0.72 MB


## Explore the detailed data.


In [5]:
def read_table_sample(table_name, limit=50):
    prefix = f"{GOLD_PATH_PREFIX}{table_name}/data/"
    try:
        objects = list(minio_client.list_objects(BUCKET_NAME, prefix=prefix, recursive=True))
        parquet_files = [obj.object_name for obj in objects if obj.object_name.endswith('.parquet')]
        
        if not parquet_files:
            print(f"[WARN] Không tìm thấy file dữ liệu cho bảng {table_name}")
            return
        
        # Đọc file đầu tiên
        response = minio_client.get_object(BUCKET_NAME, parquet_files[0])
        df = pd.read_parquet(io.BytesIO(response.read()))
        
        print(f"\n{'='*80}")
        print(f"📊 TABLE: {table_name.upper()}")
        print(f"{'='*80}")
        print(f"\n[SCHEMA]:")
        print(df.dtypes)
        print(f"\n[SAMPLE - Top {limit} rows]:")
        display(df.head(limit))
        
    except Exception as e:
        print(f"[ERROR] Lỗi khi đọc bảng {table_name}: {e}")

for table in gold_tables.keys():
    read_table_sample(table)


📊 TABLE: MART_CUSTOMER_LIFETIME_VALUE

[SCHEMA]:
customer_id            str
total_orders         int64
total_spent_usd    float64
first_order         object
last_order          object
dtype: object

[SAMPLE - Top 50 rows]:


,customer_id,total_orders,total_spent_usd,first_order,last_order
0,usr_uxaudm7j,97,289943.0301,2026-04-14,2026-04-14
1,usr_916np9zk,100,261205.4079,2026-04-14,2026-04-14
2,usr_9o3gv6qz,108,289899.1417,2026-04-14,2026-04-14
3,usr_jfxlh21w,126,378131.2240,2026-04-14,2026-04-14
4,usr_usza29fq,111,354135.6134,2026-04-14,2026-04-14
5,usr_gry37jk4,112,345672.9106,2026-04-14,2026-04-14
6,usr_r4hi0lq2,106,300023.4218,2026-04-14,2026-04-14
7,usr_lly5d8ph,105,290292.0303,2026-04-14,2026-04-14
8,usr_d3tt24eo,97,296120.7970,2026-04-14,2026-04-14
9,usr_mw29mayx,99,325742.6989,2026-04-14,2026-04-14



📊 TABLE: MART_SALES_OVERVIEW

[SCHEMA]:
order_id              str
order_date         object
customer_id           str
country               str
product_name          str
category              str
quantity            int32
amount_usd        float64
payment_status        str
dtype: object

[SAMPLE - Top 50 rows]:


,order_id,order_date,customer_id,country,product_name,category,quantity,amount_usd,payment_status
0,ord_0007ebbzuh,2026-04-14,usr_9vmvopd0,TH,Product 98,Home,1,141.5916,SETTLED
1,ord_001qx6qdna,2026-04-14,usr_8g0sa7ue,LA,Product 147,Sports,1,15.8952,SETTLED
2,ord_009rw5d76g,2026-04-14,usr_8higrhr7,MM,Product 133,Electronics,19,1528.0256,SETTLED
3,ord_009w5j03om,2026-04-14,usr_xo72480b,SG,Product 64,Electronics,1,278.8764,PENDING
4,ord_00d47ifn5v,2026-04-14,usr_tf1w0oem,KH,Product 148,Electronics,2,597.2224,SETTLED
5,ord_00enprn1nh,2026-04-14,usr_rqikzr01,VN,Product 10,Beauty,8,2285.2000,PENDING
6,ord_00ey2wzbnw,2026-04-14,usr_06ud4yqs,TH,Product 133,Electronics,2,160.8448,SETTLED
7,ord_00fga3l9jt,2026-04-14,usr_zkj1lkj8,ID,Product 124,Electronics,7,846.3000,FAILED
8,ord_00hjf87f07,2026-04-14,usr_kzznwawv,KH,Product 192,Electronics,20,1764.6000,SETTLED
9,ord_00hr2nq9t0,2026-04-14,usr_8c0or0ad,ID,Product 70,Beauty,5,1561.4000,FAILED


In [6]:
# Check for duplicates in mart_customer_lifetime_value
import io

print("="*80)
print("DUPLICATE CHECK: mart_customer_lifetime_value")
print("="*80)

try:
    # Read all parquet files from mart_customer_lifetime_value
    prefix = f"{GOLD_PATH_PREFIX}mart_customer_lifetime_value/data/"
    objects = list(minio_client.list_objects(BUCKET_NAME, prefix=prefix, recursive=True))
    parquet_files = [obj.object_name for obj in objects if obj.object_name.endswith('.parquet')]
    
    if not parquet_files:
        print(f"❌ No parquet files found for mart_customer_lifetime_value")
    else:
        print(f"\n✓ Found {len(parquet_files)} parquet files")
        
        # Read all files and combine
        dfs = []
        for pq_file in parquet_files:
            response = minio_client.get_object(BUCKET_NAME, pq_file)
            df_part = pd.read_parquet(io.BytesIO(response.read()))
            dfs.append(df_part)
        
        df_clv = pd.concat(dfs, ignore_index=True)
        
        total_records = len(df_clv)
        unique_customers = df_clv['customer_id'].nunique()
        
        print(f"\nTotal records: {total_records}")
        print(f"Unique customers: {unique_customers}")
        
        if total_records == unique_customers:
            print("\n✅ NO DUPLICATES - Each customer appears exactly once")
        else:
            print(f"\n⚠️  DUPLICATES FOUND - Extra records: {total_records - unique_customers}")
            
            # Show duplicate customers
            dup_counts = df_clv['customer_id'].value_counts()
            dupes = dup_counts[dup_counts > 1]
            
            print(f"\nCustomers with duplicates: {len(dupes)}")
            if len(dupes) > 0:
                print("\nTop 10 duplicated customers:")
                print(dupes.head(10))
        
        print("\nSample records from mart_customer_lifetime_value:")
        display(df_clv.head(10))
        
except Exception as e:
    print(f"❌ Error: {e}")

DUPLICATE CHECK: mart_customer_lifetime_value

✓ Found 1 parquet files

Total records: 500
Unique customers: 500

✅ NO DUPLICATES - Each customer appears exactly once

Sample records from mart_customer_lifetime_value:


,customer_id,total_orders,total_spent_usd,first_order,last_order
0,usr_uxaudm7j,97,289943.0301,2026-04-14,2026-04-14
1,usr_916np9zk,100,261205.4079,2026-04-14,2026-04-14
2,usr_9o3gv6qz,108,289899.1417,2026-04-14,2026-04-14
3,usr_jfxlh21w,126,378131.2240,2026-04-14,2026-04-14
4,usr_usza29fq,111,354135.6134,2026-04-14,2026-04-14
5,usr_gry37jk4,112,345672.9106,2026-04-14,2026-04-14
6,usr_r4hi0lq2,106,300023.4218,2026-04-14,2026-04-14
7,usr_lly5d8ph,105,290292.0303,2026-04-14,2026-04-14
8,usr_d3tt24eo,97,296120.7970,2026-04-14,2026-04-14
9,usr_mw29mayx,99,325742.6989,2026-04-14,2026-04-14


In [7]:
# Check for duplicates in mart_sales_overview
import io

print("="*80)
print("DUPLICATE CHECK: mart_sales_overview")
print("="*80)

try:
    # Read all parquet files from mart_sales_overview
    prefix = f"{GOLD_PATH_PREFIX}mart_sales_overview/data/"
    objects = list(minio_client.list_objects(BUCKET_NAME, prefix=prefix, recursive=True))
    parquet_files = [obj.object_name for obj in objects if obj.object_name.endswith('.parquet')]
    
    if not parquet_files:
        print(f"❌ No parquet files found for mart_sales_overview")
    else:
        print(f"\n✓ Found {len(parquet_files)} parquet files")
        
        # Read all files and combine
        dfs = []
        for pq_file in parquet_files:
            response = minio_client.get_object(BUCKET_NAME, pq_file)
            df_part = pd.read_parquet(io.BytesIO(response.read()))
            dfs.append(df_part)
        
        df_sales = pd.concat(dfs, ignore_index=True)
        
        total_records = len(df_sales)
        unique_orders = df_sales['order_id'].nunique()
        
        print(f"\nTotal records: {total_records}")
        print(f"Unique orders: {unique_orders}")
        
        if total_records == unique_orders:
            print("\n✅ NO DUPLICATES - Each order appears exactly once")
        else:
            print(f"\n⚠️  DUPLICATES FOUND - Extra records: {total_records - unique_orders}")
            
            # Show duplicate orders
            dup_counts = df_sales['order_id'].value_counts()
            dupes = dup_counts[dup_counts > 1]
            
            print(f"\nOrders with duplicates: {len(dupes)}")
            if len(dupes) > 0:
                print("\nTop 10 duplicated orders:")
                print(dupes.head(10))
        
        print("\nSample records from mart_sales_overview:")
        display(df_sales.head(10))
        
except Exception as e:
    print(f"❌ Error: {e}")

DUPLICATE CHECK: mart_sales_overview

✓ Found 1 parquet files

Total records: 52966
Unique orders: 52966

✅ NO DUPLICATES - Each order appears exactly once

Sample records from mart_sales_overview:


,order_id,order_date,customer_id,country,product_name,category,quantity,amount_usd,payment_status
0,ord_0007ebbzuh,2026-04-14,usr_9vmvopd0,TH,Product 98,Home,1,141.5916,SETTLED
1,ord_001qx6qdna,2026-04-14,usr_8g0sa7ue,LA,Product 147,Sports,1,15.8952,SETTLED
2,ord_009rw5d76g,2026-04-14,usr_8higrhr7,MM,Product 133,Electronics,19,1528.0256,SETTLED
3,ord_009w5j03om,2026-04-14,usr_xo72480b,SG,Product 64,Electronics,1,278.8764,PENDING
4,ord_00d47ifn5v,2026-04-14,usr_tf1w0oem,KH,Product 148,Electronics,2,597.2224,SETTLED
5,ord_00enprn1nh,2026-04-14,usr_rqikzr01,VN,Product 10,Beauty,8,2285.2000,PENDING
6,ord_00ey2wzbnw,2026-04-14,usr_06ud4yqs,TH,Product 133,Electronics,2,160.8448,SETTLED
7,ord_00fga3l9jt,2026-04-14,usr_zkj1lkj8,ID,Product 124,Electronics,7,846.3000,FAILED
8,ord_00hjf87f07,2026-04-14,usr_kzznwawv,KH,Product 192,Electronics,20,1764.6000,SETTLED
9,ord_00hr2nq9t0,2026-04-14,usr_8c0or0ad,ID,Product 70,Beauty,5,1561.4000,FAILED


In [17]:
# Check for NULL values in Gold tables
import io

print("=" * 80)
print("NULL CHECK: mart_sales_overview")
print("=" * 80)

try:
    # Read all parquet files from mart_sales_overview
    prefix = f"{GOLD_PATH_PREFIX}mart_sales_overview/data/"
    objects = list(minio_client.list_objects(BUCKET_NAME, prefix=prefix, recursive=True))
    parquet_files = [obj.object_name for obj in objects if obj.object_name.endswith('.parquet')]
    
    if not parquet_files:
        print(f"❌ No parquet files found for mart_sales_overview")
    else:
        print(f"\n✓ Found {len(parquet_files)} parquet files")
        
        # Read all files and combine
        dfs = []
        for pq_file in parquet_files:
            response = minio_client.get_object(BUCKET_NAME, pq_file)
            df_part = pd.read_parquet(io.BytesIO(response.read()))
            dfs.append(df_part)
        
        df_sales = pd.concat(dfs, ignore_index=True)
        
        print(f"\nTotal rows: {len(df_sales)}")
        print(f"Columns: {list(df_sales.columns)}\n")
        
        # Check for NULL in each column
        columns_to_check = ["order_id", "order_date", "customer_id", "country", "product_name", "category", "quantity", "amount_usd", "payment_status"]
        
        print("NULL Count by Column:")
        print("-" * 40)
        has_nulls = False
        for col_name in columns_to_check:
            if col_name in df_sales.columns:
                null_count = df_sales[col_name].isna().sum()
                total = len(df_sales)
                pct = (null_count / total * 100) if total > 0 else 0
                print(f"{col_name:20} NULL: {null_count:6} ({pct:.1f}%)")
                if null_count > 0:
                    has_nulls = True
        
        if not has_nulls:
            print("\n✅ NO NULL VALUES - All columns have complete data!")
        else:
            # Show details for columns with NULLs
            print("\n" + "="*80)
            print("DETAILS FOR COLUMNS WITH NULL VALUES")
            print("="*80)
            
            for col_name in columns_to_check:
                if col_name in df_sales.columns and df_sales[col_name].isna().sum() > 0:
                    print(f"\n### {col_name} ###")
                    print(f"Sample rows with NULL {col_name}:")
                    null_rows = df_sales[df_sales[col_name].isna()].head(3)
                    print(null_rows[["order_id", col_name]].to_string())
        
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

NULL CHECK: mart_sales_overview

✓ Found 1 parquet files

Total rows: 52966
Columns: ['order_id', 'order_date', 'customer_id', 'country', 'product_name', 'category', 'quantity', 'amount_usd', 'payment_status']

NULL Count by Column:
----------------------------------------
order_id             NULL:      0 (0.0%)
order_date           NULL:      0 (0.0%)
customer_id          NULL:      0 (0.0%)
country              NULL:      0 (0.0%)
product_name         NULL:      0 (0.0%)
category             NULL:      0 (0.0%)
quantity             NULL:      0 (0.0%)
amount_usd           NULL:      0 (0.0%)
payment_status       NULL:      0 (0.0%)

✅ NO NULL VALUES - All columns have complete data!
